# Host–Guest Committor: Mechanistic Insight

Maps the RPE free-energy landscape, shooting outcomes, learned committor, and descriptor distributions to expose the host–guest transition mechanism.

This notebook is independently runnable: it imports its dependencies, resolves
the project paths, loads the RPE trainset and trained committor model, creates
the `SystemVisualizer`, and restores or computes the model-output cache.


## 1. Imports and shared analysis helpers


In [ ]:
import os
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import aimmd
import aimmd.pytorch
import aimmd.pytorch.rcmodel
import aimmdTIS
from aimmdTIS import Tools as TAI_Tools
from aimmdTIS.visualization import SystemVisualizer

plt.rcParams.update({
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.labelsize": 16,
})


## 2. Project and model configuration


In [ ]:
DEFAULT_WORK_DIR = Path.cwd()
WORK_DIR = Path(os.environ.get("HG_COMMITTOR_WORK_DIR", DEFAULT_WORK_DIR)).expanduser().resolve()
ANALYSIS_DIR = WORK_DIR

if not ANALYSIS_DIR.is_dir():
    raise FileNotFoundError(
        f"Analysis directory not found: {ANALYSIS_DIR}. "
        "Set HG_COMMITTOR_WORK_DIR to the project root."
    )
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from committor_analysis_helpers import (
    aimmd_softplus_loss_terms,
    conditional_permutation_importance,
    descriptor_names,
    descriptor_swap_loss_matrix,
    permutation_influence_correlation,
    plot_correlated_pair,
    plot_square_matrix,
    predict_q,
    run_descriptor_correlation_tools,
    sigmoid_stable,
    subsample_arrays,
    weighted_gradient_importance,
    weighted_permutation_importance,
)

TRAINSET_FILE = WORK_DIR / "Files" / "trainset_rpe.pkl"
MODEL_DIR = WORK_DIR / "Files"
MODEL_FILE = MODEL_DIR / "aimmd_store_s5em04.h5"
MODEL_KEY = "stage1_full_model_model_best"
OUTPUT_DIR = WORK_DIR / "Figures"
CACHE_DIR = MODEL_DIR / ".cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for required_file in (TRAINSET_FILE, MODEL_FILE):
    if not required_file.is_file():
        raise FileNotFoundError(required_file)

DESCRIPTOR_LABELS = [
    "Host-Guest Distance",
    "Guest Orientation Angle",
    "Guest Position Angle",
    "Hydrogen Bonds",
    "Waters in Cavity",
    "Hydrophobic Contact Score",
    "Shared Waters",
]
DESCRIPTOR_UNITS = ["[nm]", "[rad]", "[rad]", "[-]", "[-]", "[-]", "[-]"]
DESC_MIN = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
DESC_MAX = np.array([1.0, np.pi, np.pi, 4.0, 10.0, 60.0, 10.0])
N_BINS_DIM = np.array([200, 200, 200, 6, 50, 200, 50])
STATE_A, STATE_B = 0.05, 1.0
Q_MIN, Q_MAX = -45.0, 15.0
N_DESC = len(DESCRIPTOR_LABELS)


## 3. Load the RPE trainset and committor model


In [ ]:
clear_unbound = True

with TRAINSET_FILE.open("rb") as handle:
    trainset = pickle.load(handle)

required_keys = {"descriptors", "weights", "shot_results"}
missing_keys = required_keys.difference(trainset)
if missing_keys:
    raise KeyError(f"Trainset is missing keys: {sorted(missing_keys)}")

mask_clear_unbound = trainset["descriptors"][:, 0] <= 1.1 if clear_unbound else np.ones(len(trainset["descriptors"]), dtype=bool)

descriptors = np.asarray(trainset["descriptors"][mask_clear_unbound], dtype=float)
weights = np.asarray(trainset["weights"][mask_clear_unbound], dtype=float).reshape(-1)
shot_results = np.asarray(trainset["shot_results"][mask_clear_unbound])

if descriptors.ndim != 2 or descriptors.shape[1] != N_DESC:
    raise ValueError(f"Expected descriptors with shape (N, {N_DESC}), got {descriptors.shape}.")
if len(weights) != len(descriptors) or shot_results.shape != (len(descriptors), 2):
    raise ValueError("Trainset descriptors, weights, and shot results are not aligned.")
if np.any(weights < 0) or not np.all(np.isfinite(weights)):
    raise ValueError("Trainset weights must be finite and non-negative.")

positive_weights = weights[weights > 0]
if positive_weights.size == 0:
    raise ValueError("Trainset contains no positive weights.")
weights = weights / positive_weights.min()
trainset["weights"] = weights
trainset["descriptors"] = descriptors
trainset["shot_results"] = shot_results

storage = aimmd.Storage(str(MODEL_FILE), mode="r")
if MODEL_KEY not in storage.rcmodels:
    raise KeyError(f"Model key {MODEL_KEY!r} not found; available: {list(storage.rcmodels)}")
model = storage.rcmodels[MODEL_KEY]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TAI_Tools.model_to(model, device)

print(f"Frames: {len(descriptors):,}; descriptors: {descriptors.shape[1]}")
print(f"Shot results: {shot_results.shape[0]:,} frames, {shot_results.shape[1]} shots each")
print(f"weights: {weights.shape[0]:,} frames, min={weights.min():.3g}, max={weights.max():.3g}")
print(f"Weight range: {weights.min():.3g} – {weights.max():.3g}")
print(f"Model: {MODEL_KEY} on {device}")


## 4. Create and initialize the `SystemVisualizer`


In [ ]:
standard_value = np.average(descriptors, axis=0).tolist()
vis = SystemVisualizer(
    temperature=1.0,
    dims_extent=[0.0, 1.2, 0.0, float(np.pi)],
    total_num_descriptors=N_DESC,
    standard_value=standard_value,
    descriptor_labels=DESCRIPTOR_LABELS,
    desc_min=DESC_MIN,
    desc_max=DESC_MAX,
)
vis.load_trainset(trainset)

print(f"Visualizer descriptors: {vis._desc.shape}")
print(f"Visualizer physical descriptors: {vis._desc_phys.shape}")


## 5. Restore or compute model outputs


In [ ]:
# Separate cache when unbound frames are removed (clear_unbound), since the frame set differs
Q_CACHE_FILE = CACHE_DIR / ("q_model_mbar_clear_unbound.npy" if clear_unbound else "q_model_mbar.npy")
RECOMPUTE_Q_MODEL = False  # set to True after changing the model or trainset
if Q_CACHE_FILE.is_file() and not RECOMPUTE_Q_MODEL:
    q_model = np.load(Q_CACHE_FILE)
    if q_model.shape != (len(descriptors),):
        raise ValueError(
            f"Cached model output has shape {q_model.shape}; expected {(len(descriptors),)}. "
            "Delete the stale cache and rerun this cell."
        )
    p_B = sigmoid_stable(q_model)
    vis._model_output_cache[model] = (p_B, q_model)
    print(f"Loaded model output from {Q_CACHE_FILE}")
else:
    p_B, q_model = vis._model_output_rpe(model)
    np.save(Q_CACHE_FILE, q_model)
    print(f"Computed and cached model output at {Q_CACHE_FILE}")

print(f"q range: {q_model.min():.3f} – {q_model.max():.3f}")


## 6. Stable-state occupancy and model-derived interfaces


In [ ]:
in_state_A = (vis._desc_phys[:, 0] < STATE_A)  # dim0 physical < 0.05 nm
in_state_B = (vis._desc_phys[:, 0] > STATE_B)  # dim0 physical > 1.0 nm

# Correct orientation: (n_samples, n_states)
in_state = np.column_stack([in_state_A, in_state_B])

print('A count:', int(in_state[:, 0].sum()))
print('B count:', int(in_state[:, 1].sum()))

out_interfaces = aimmdTIS.Tools.check_interfaces(
    model,
    ('A', 'B'),
    vis._desc,
    in_state=in_state,
    overlap=0.3,
)

print('min_max_stable_q:\n', out_interfaces['min_max_stable_q'])
print('forward interfaces:', out_interfaces['forward_interfaces'])
print('backward interfaces:', out_interfaces['backward_interfaces'])


## 7. RPE-weighted free-energy surfaces


In [ ]:
fe_contour_levels = [2, 5, 10, 15, 20, 30, 40,41, 42, 43]
v_min_max_fe      = (0, 50)          # colour scale for ΔF

n_other = N_DESC - 1
ncols, nrows = 3, 2
fig, axarr = plt.subplots(
    nrows, ncols, figsize=(ncols * 6, nrows * 5),
    gridspec_kw={"wspace": 0.4, "hspace": 0.5},
)
axarr = axarr.flatten()

for i, other_dim in enumerate(range(1, N_DESC)):
    ax    = axarr[i]
    dims  = (0, other_dim)
    n_bins_2d = (int(N_BINS_DIM[0]), int(N_BINS_DIM[other_dim]))

    # Use actual data range for extent so empty whitespace is avoided
    actual_x_max = 1.2
    actual_y_min = float(vis._desc_phys[:, other_dim].min())
    actual_y_max = float(vis._desc_phys[:, other_dim].max())

    # Update extent for this pair in PHYSICAL space (x = dim0, y = other_dim)
    vis.dims_extent = [
        float(DESC_MIN[0]),         actual_x_max,
        actual_y_min,               actual_y_max,
    ]
    vis.descriptor_dims = dims

    # Filled free-energy heatmap  (new method — uses vis._desc_phys, vis._w)
    im = vis.plot_rpe_free_energy(
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        offset=True,
        v_min_max=v_min_max_fe,
        cmap="Blues",
    )

    # Isocontours (skip for very discrete descriptors)
    # Reuse the histogram already stored in vis._cache
    n_bins_2d_tuple = (n_bins_2d, n_bins_2d) if np.shape(n_bins_2d) == () else tuple(n_bins_2d)
    extent_key = tuple(vis.dims_extent)
    cache_key  = f"fe2d_{dims}_{n_bins_2d_tuple[0]}_{n_bins_2d_tuple[1]}_{extent_key}"
    H  = vis._cache[cache_key]
    xedges, yedges = vis.create_x_y_edges(n_bins_2d=n_bins_2d)
    with np.errstate(divide="ignore"):
        fe = np.where(H > 0, -np.log(H), np.nan)
    fe -= np.nanmin(fe[np.isfinite(fe)])
    xc = (xedges[:-1] + xedges[1:]) / 2
    yc = (yedges[:-1] + yedges[1:]) / 2
    Xc, Yc = np.meshgrid(xc, yc)
    contour = ax.contour(
        Xc, Yc, fe.T,
        levels=fe_contour_levels,
        colors="white", linewidths=0.8, alpha=0.7,
    )
    ax.clabel(contour, inline=1, fontsize=vis.plot_settings.fontsize * 0.6)

    # vlines span full y-axis regardless of physical y-range (xaxis transform)
    ax.vlines(
        [STATE_A, STATE_B], 0, 1,
        transform=ax.get_xaxis_transform(),
        linewidths=1.5, colors="black", linestyles="--",
    )
    ax.set_xlim(float(DESC_MIN[0]), actual_x_max)
    ax.set_ylim(actual_y_min, actual_y_max)
    ax.set_xlabel(f"{DESCRIPTOR_LABELS[0]} {DESCRIPTOR_UNITS[0]}", fontsize=11)
    ax.set_ylabel(f"{DESCRIPTOR_LABELS[other_dim]} {DESCRIPTOR_UNITS[other_dim]}", fontsize=11)

for j in range(n_other, len(axarr)):
    fig.delaxes(axarr[j])

cbar_ax = fig.add_axes([0.25, 0.97, 0.5, 0.018])
fig.colorbar(im, cax=cbar_ax, orientation="horizontal").set_label(
    r"$\Delta F = -\ln(\rho_{\rm RPE})$ [arbitrary units]", labelpad=6
)
plt.suptitle("MBAR-weighted free-energy surface (RPE) — SystemVisualizer", y=1.04, fontsize=14)
TAI_Tools.save_fig_pdf_and_png(fig, "HG_free_energy_2d_newvis", output_path=OUTPUT_DIR)
plt.show()


## 8. Shooting-data committor surfaces


In [ ]:
levels_q    = [-45,-30, -20, -10, -5, -3, -1, 0, 1,5, 10, 15]
v_min_max_q = (-45, 15)

n_other = N_DESC - 1
ncols, nrows = 3, 2
fig, axarr = plt.subplots(
    nrows, ncols, figsize=(ncols * 6, nrows * 5),
    gridspec_kw={"wspace": 0.4, "hspace": 0.5},
)
axarr = axarr.flatten()

for i, other_dim in enumerate(range(1, N_DESC)):
    ax    = axarr[i]
    dims  = (0, other_dim)
    n_bins_2d = (int(N_BINS_DIM[0]), int(N_BINS_DIM[other_dim]))

    # Use actual data range for extent so empty whitespace is avoided
    actual_x_max = 1.2
    actual_y_min = float(vis._desc_phys[:, other_dim].min())
    actual_y_max = float(vis._desc_phys[:, other_dim].max())

    # Update extent for this pair in PHYSICAL space (x = dim0, y = other_dim)
    vis.dims_extent = [
        float(DESC_MIN[0]),         actual_x_max ,
        actual_y_min,               actual_y_max,
    ]
    vis.descriptor_dims = dims

    # Filled 2-D committor from shooting data  (new method — no model call)
    im = vis.plot_shot_committor(
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        v_min_max=v_min_max_q,
        cmap="Spectral",
    )

    # # Model q contour lines (cached grid: free after first call)
    # vis.plot_q_contours(
    #     model,
    #     ax=ax,
    #     descriptor_dims=dims,
    #     n_bins_2d=200,
    #     levels=levels_q,
    #     colors="black",
    #     linewidths=0.8,
    #     alpha=0.6,
    # )
    vis.plot_shot_committor_contours(
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        levels=levels_q,
        colors="black",
        linewidths=0.8,
        alpha=0.75,
        smooth_sigma=0.5,
        clabel=True,
    )
        

    # vlines span full y-axis regardless of physical y-range (xaxis transform)
    ax.vlines(
        [STATE_A, STATE_B], 0, 1,
        transform=ax.get_xaxis_transform(),
        linewidths=1.5, colors="white", linestyles="--",
    )
    ax.set_xlim(float(DESC_MIN[0]), actual_x_max)
    ax.set_ylim(actual_y_min, actual_y_max)
    ax.set_xlabel(f"{DESCRIPTOR_LABELS[0]} {DESCRIPTOR_UNITS[0]}", fontsize=11)
    ax.set_ylabel(f"{DESCRIPTOR_LABELS[other_dim]} {DESCRIPTOR_UNITS[other_dim]}", fontsize=11)

for j in range(n_other, len(axarr)):
    fig.delaxes(axarr[j])

cbar_ax = fig.add_axes([0.25, 0.97, 0.5, 0.018])
fig.colorbar(im, cax=cbar_ax, orientation="horizontal").set_label(
    r"$q = \ln(\bar{n}_B / \bar{n}_A)$ from shot results", labelpad=6
)
plt.suptitle(
    "Data committor (shot results) with model $q$ contours — SystemVisualizer",
    y=1.04, fontsize=14,
)
TAI_Tools.save_fig_pdf_and_png(fig, "HG_q_committor_shots_newvis", output_path=OUTPUT_DIR)
plt.show()


## 9. Model-logit committor surfaces


In [ ]:
# ── 10a: Logit committor q = ln(pB/pA) projected onto descriptor space ────────
# Background: MBAR-weighted average model logit committor per bin.
# Contour overlay 1 (black solid):  model projection contours — same data as background.
# Contour overlay 2 (grey dashed):  data-estimated committor from shot results.
v_min_max_proj_q = (-45, 15)
contour_levels_q = [-45,-30,-15, -10, -5, -3, -1, 0, 1, 3, 5, 10, 15]

n_other = N_DESC - 1
ncols, nrows = 3, 2
fig_qproj, axarr_qproj = plt.subplots(
    nrows, ncols, figsize=(ncols * 6, nrows * 5),
    gridspec_kw={"wspace": 0.4, "hspace": 0.5},
)
axarr_qproj = axarr_qproj.flatten()

for i, other_dim in enumerate(range(1, N_DESC)):
    ax    = axarr_qproj[i]
    dims  = (0, other_dim)
    n_bins_2d = (int(N_BINS_DIM[0]), int(N_BINS_DIM[other_dim]))

    actual_x_max = 1.2
    actual_y_min = float(vis._desc_phys[:, other_dim].min())
    actual_y_max = float(vis._desc_phys[:, other_dim].max())

    vis.dims_extent = [float(DESC_MIN[0]), actual_x_max, actual_y_min, actual_y_max]
    vis.descriptor_dims = dims

    # Background heatmap
    im = vis.plot_q_model_projection(
        model,
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        v_min_max=v_min_max_proj_q,
        cmap="Spectral",
        logit=True,
    )
    # Contour overlay: model projection iso-q lines (reuses cache, no extra forward pass)
    vis.plot_q_model_projection_contours(
        model,
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        levels=contour_levels_q,
        logit=True,
        colors="black",
        linewidths=0.7,
        alpha=0.6,
        linestyles="solid",
        clabel=True,
        clabel_fontsize=6,
    )
    # Contour overlay: data-based shot committor (dashed grey)
    vis.plot_shot_committor_contours(
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        levels=contour_levels_q,
        colors="dimgray",
        linewidths=0.9,
        alpha=0.75,
        linestyles="dashed",
        smooth_sigma=1.5,
        clabel=False,
    )
    ax.vlines(
        [STATE_A, STATE_B], 0, 1,
        transform=ax.get_xaxis_transform(),
        linewidths=1.5, colors="black", linestyles="--",
    )
    ax.set_xlim(float(DESC_MIN[0]), actual_x_max)
    ax.set_ylim(actual_y_min, actual_y_max)
    ax.set_xlabel(f"{DESCRIPTOR_LABELS[0]} {DESCRIPTOR_UNITS[0]}", fontsize=11)
    ax.set_ylabel(f"{DESCRIPTOR_LABELS[other_dim]} {DESCRIPTOR_UNITS[other_dim]}", fontsize=11)

for j in range(n_other, len(axarr_qproj)):
    fig_qproj.delaxes(axarr_qproj[j])

cbar_ax = fig_qproj.add_axes([0.25, 0.97, 0.5, 0.018])
fig_qproj.colorbar(im, cax=cbar_ax, orientation="horizontal").set_label(
    r"$\langle q(x|\theta) \rangle_{\rm RPE}$ [logit committor]", labelpad=6
)
# Legend proxy patches
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
leg_model  = mlines.Line2D([], [], color="black",   lw=0.7, ls="solid",  label="model projection contours")
leg_data   = mlines.Line2D([], [], color="dimgray", lw=0.9, ls="dashed", label="shot-data committor contours")
fig_qproj.legend(handles=[leg_model, leg_data], loc="lower center",
                 bbox_to_anchor=(0.5, -0.04), ncol=2, fontsize=10, frameon=False)
plt.suptitle("MBAR-weighted model logit committor projection — SystemVisualizer", y=1.04, fontsize=14)
TAI_Tools.save_fig_pdf_and_png(fig_qproj, "HG_q_model_projection_logit_newvis", output_path=OUTPUT_DIR)
plt.show()


## 10. Model-probability committor surfaces


In [ ]:
# ── 10b: Linear committor pB projected onto descriptor space ──────────────────
# Background: MBAR-weighted average model pB per bin.
# Contour overlay 1 (black solid):  model projection pB iso-lines.
# Contour overlay 2 (grey dashed):  data-based committor contours (logit → pB levels mapped).
v_min_max_pb = (0.0, 1.0)
contour_levels_pB  = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
# For the shot-committor overlay on the pB plot, map pB levels to logit space
contour_levels_shot_for_pB = [
    float(np.log(p / (1 - p))) for p in contour_levels_pB if 0 < p < 1
]

n_other = N_DESC - 1
ncols, nrows = 3, 2
fig_pbproj, axarr_pbproj = plt.subplots(
    nrows, ncols, figsize=(ncols * 6, nrows * 5),
    gridspec_kw={"wspace": 0.4, "hspace": 0.5},
)
axarr_pbproj = axarr_pbproj.flatten()

for i, other_dim in enumerate(range(1, N_DESC)):
    ax    = axarr_pbproj[i]
    dims  = (0, other_dim)
    n_bins_2d = (int(N_BINS_DIM[0]), int(N_BINS_DIM[other_dim]))

    actual_x_max = 1.2
    actual_y_min = float(vis._desc_phys[:, other_dim].min())
    actual_y_max = float(vis._desc_phys[:, other_dim].max())

    vis.dims_extent = [float(DESC_MIN[0]), actual_x_max, actual_y_min, actual_y_max]
    vis.descriptor_dims = dims

    # Background heatmap
    im = vis.plot_q_model_projection(
        model,
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        v_min_max=v_min_max_pb,
        cmap="RdYlGn",
        logit=False,
    )
    # Contour overlay: model pB iso-lines (reuses cache)
    vis.plot_q_model_projection_contours(
        model,
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        levels=contour_levels_pB,
        logit=False,
        colors="black",
        linewidths=0.7,
        alpha=0.6,
        linestyles="solid",
        clabel=True,
        clabel_fontsize=6,
    )
    # Contour overlay: data-based shot committor in logit space → same iso-surfaces
    vis.plot_shot_committor_contours(
        descriptor_dims=dims,
        n_bins_2d=n_bins_2d,
        ax=ax,
        levels=contour_levels_shot_for_pB,
        colors="dimgray",
        linewidths=0.9,
        alpha=0.75,
        linestyles="dashed",
        smooth_sigma=1.5,
        clabel=False,
    )
    ax.vlines(
        [STATE_A, STATE_B], 0, 1,
        transform=ax.get_xaxis_transform(),
        linewidths=1.5, colors="black", linestyles="--",
    )
    ax.set_xlim(float(DESC_MIN[0]), actual_x_max)
    ax.set_ylim(actual_y_min, actual_y_max)
    ax.set_xlabel(f"{DESCRIPTOR_LABELS[0]} {DESCRIPTOR_UNITS[0]}", fontsize=11)
    ax.set_ylabel(f"{DESCRIPTOR_LABELS[other_dim]} {DESCRIPTOR_UNITS[other_dim]}", fontsize=11)

for j in range(n_other, len(axarr_pbproj)):
    fig_pbproj.delaxes(axarr_pbproj[j])

cbar_ax = fig_pbproj.add_axes([0.25, 0.97, 0.5, 0.018])
fig_pbproj.colorbar(im, cax=cbar_ax, orientation="horizontal").set_label(
    r"$\langle p_B(x|\theta) \rangle_{\rm RPE}$", labelpad=6
)
import matplotlib.lines as mlines
leg_model = mlines.Line2D([], [], color="black",   lw=0.7, ls="solid",  label=r"model $p_B$ contours")
leg_data  = mlines.Line2D([], [], color="dimgray", lw=0.9, ls="dashed", label="shot-data committor contours")
fig_pbproj.legend(handles=[leg_model, leg_data], loc="lower center",
                  bbox_to_anchor=(0.5, -0.04), ncol=2, fontsize=10, frameon=False)
plt.suptitle(r"MBAR-weighted model $p_B$ projection — SystemVisualizer", y=1.04, fontsize=14)
TAI_Tools.save_fig_pdf_and_png(fig_pbproj, "HG_q_model_projection_pB_newvis", output_path=OUTPUT_DIR)
plt.show()


## 11. Descriptor distributions along the committor


In [ ]:
descriptors_show = [0,1,4]
n_cols = 1
n_rows = len(descriptors_show) // n_cols + int(len(descriptors_show) % n_cols > 0)
fig,ax = plt.subplots(n_rows, n_cols, figsize=(16, 12), sharex=True, sharey=False)
ax = ax.flatten()
xedges = np.arange(-45,15,0.1)

use_weights = True
vmax = -3
vmin= -15
num_model_descriptors = len(DESCRIPTOR_LABELS)
for i, i_dim in enumerate(descriptors_show):
    desc = vis._desc_phys[:, i_dim]
    yedges = np.linspace(np.min(desc), np.max(desc), N_BINS_DIM[i_dim])
    H_q_descriptors, x_bins, y_bins = np.histogram2d(q_model[:], desc[:],
                                        bins=[xedges, yedges], weights=vis._w if use_weights else None, density=False)
    for j in range(H_q_descriptors.shape[0]):
        row_sum = H_q_descriptors[j].sum()
        if row_sum > 0:
            H_q_descriptors[j] /= row_sum  # Normalize each row to sum to 1
    im = ax[i].imshow(np.log(H_q_descriptors.T), 
                      origin='lower', 
                      cmap='Blues', 
                      extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], 
                      aspect='auto',
                      vmax=vmax, vmin=vmin,
                      interpolation="gaussian",)
    fig.colorbar(im, ax=ax[i],  pad=0.03, label=r'$\ln \rho(\xi_i|q)$')
    ind_range = np.where(H_q_descriptors>0)[1]
    y_max = np.max(y_bins[ind_range]) if len(ind_range)>0 else DESC_MAX[i_dim]
    ax[i].grid(True, linestyle='--', alpha=0.5)
    ax[i].set_ylim([np.min(y_bins[ind_range]), y_max])
    desc_unit = f"{DESCRIPTOR_UNITS[i_dim]}" if DESCRIPTOR_UNITS[i_dim] else ""
    ax[i].set_ylabel(DESCRIPTOR_LABELS[i_dim] + " " + desc_unit, fontsize=16)
for i in range(num_model_descriptors, len(ax)):
    fig.delaxes(ax[i])
ax[i].set_xlabel(r"$q(x|\theta)$", fontsize=20)

# plt.tight_layout()

from matplotlib.transforms import ScaledTranslation

labels = ["A", "B", "C"]

offset_x = -15  # pixels
offset_y = 5   # pixels

for i, ax in enumerate(ax):

    trans = ax.transAxes + ScaledTranslation(
        offset_x / fig.dpi,
        offset_y / fig.dpi,
        fig.dpi_scale_trans
    )

    ax.text(
        0, 1,
        labels[i],
        transform=trans,
        fontsize=20,
        fontweight="bold",
        va="bottom",
        ha="right",
    )

fig.align_ylabels()
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_q_model_projection_descriptors_newvis", output_path=OUTPUT_DIR)


## 12. Four-view mechanistic comparison

This final synthesis figure places four complementary views side by side for every host-distance/descriptor pair: the RPE free-energy landscape, the shot-derived logit committor, the RPE-projected model committor, and the same projection overlaid with fixed-coordinate model slices. Existing visualizer caches are reused wherever possible.


In [ ]:
comparison_dims = list(range(1, N_DESC))
column_titles = [
    "RPE free energy + contours",
    "Shot committor (logit) + contours",
    "Model projection + projection contours",
    "Model projection + fixed-coordinate slices",
]
q_levels_summary = [-30, -15, -10, -5, -3, -1, 0, 1, 3, 5, 10, 15]
fe_levels_summary = [2, 5, 10, 15, 20, 30, 40]

fig_summary, axes_summary = plt.subplots(
    len(comparison_dims),
    4,
    figsize=(24, 4.2 * len(comparison_dims)),
    constrained_layout=True,
    squeeze=False,
)
summary_images = [None] * 4

for row, other_dim in enumerate(comparison_dims):
    dims = (0, other_dim)
    bins = (int(N_BINS_DIM[0]), int(N_BINS_DIM[other_dim]))
    y_min = float(vis._desc_phys[:, other_dim].min())
    y_max = float(vis._desc_phys[:, other_dim].max())
    vis.dims_extent = [float(DESC_MIN[0]), 1.2, y_min, y_max]
    vis.descriptor_dims = dims

    # 1. RPE free energy with its own iso-free-energy contours.
    ax = axes_summary[row, 0]
    summary_images[0] = vis.plot_rpe_free_energy(
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        offset=True,
        v_min_max=(0, 50),
        cmap="Blues",
    )
    vis.plot_rpe_free_energy_contours(
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        levels=fe_levels_summary,
        colors="white",
        linewidths=0.8,
        alpha=0.8,
        smooth_sigma=1.0,
        clabel=False,
    )

    # 2. Shooting-result committor in logit space with shot-data contours.
    ax = axes_summary[row, 1]
    summary_images[1] = vis.plot_shot_committor(
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        v_min_max=(-45, 15),
        cmap="Spectral",
    )
    vis.plot_shot_committor_contours(
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        levels=q_levels_summary,
        colors="black",
        linewidths=0.8,
        alpha=0.75,
        smooth_sigma=1.5,
        clabel=False,
    )

    # 3. RPE-projected model q with contours of the projected model q.
    ax = axes_summary[row, 2]
    summary_images[2] = vis.plot_q_model_projection(
        model,
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        v_min_max=(-45, 15),
        cmap="Spectral",
        logit=True,
    )
    vis.plot_q_model_projection_contours(
        model,
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        levels=q_levels_summary,
        logit=True,
        colors="black",
        linewidths=0.8,
        alpha=0.75,
        smooth_sigma=1.0,
        clabel=False,
    )

    # 4. The same projection with direct model slices. Other descriptors are
    # held at standard_value, making the distinction from the RPE projection explicit.
    ax = axes_summary[row, 3]
    summary_images[3] = vis.plot_q_model_projection(
        model,
        descriptor_dims=dims,
        n_bins_2d=bins,
        ax=ax,
        v_min_max=(-45, 15),
        cmap="Spectral",
        logit=True,
    )
    vis.plot_q_contours(
        model,
        ax=ax,
        descriptor_dims=dims,
        n_bins_2d=bins,
        standard_value=standard_value,
        levels=q_levels_summary,
        colors="black",
        linewidths=0.8,
        alpha=0.8,
    )

    for column, panel in enumerate(axes_summary[row]):
        panel.vlines(
            [STATE_A, STATE_B],
            0,
            1,
            transform=panel.get_xaxis_transform(),
            colors="dimgray" if column == 0 else "white",
            linestyles="--",
            linewidths=1.2,
        )
        panel.set_xlim(float(DESC_MIN[0]), 1.2)
        panel.set_ylim(y_min, y_max)
        panel.set_xlabel(f"{DESCRIPTOR_LABELS[0]} [{DESCRIPTOR_UNITS[0]}]")
        panel.set_ylabel(
            f"{DESCRIPTOR_LABELS[other_dim]}"
            + (f" [{DESCRIPTOR_UNITS[other_dim]}]" if DESCRIPTOR_UNITS[other_dim] else "")
        )
        if row == 0:
            panel.set_title(column_titles[column], fontsize=12)

colorbar_labels = [
    r"$\Delta F=-\ln\rho_{\mathrm{RPE}}$",
    r"shot-derived $q=\ln(p_B/p_A)$",
    r"$\langle q(x|\theta)\rangle_{\mathrm{RPE}}$",
    r"$\langle q(x|\theta)\rangle_{\mathrm{RPE}}$",
]
for column, (image, label) in enumerate(zip(summary_images, colorbar_labels)):
    fig_summary.colorbar(
        image,
        ax=axes_summary[:, column],
        orientation="horizontal",
        shrink=0.82,
        pad=0.02,
        label=label,
    )

fig_summary.suptitle(
    "Host–guest mechanism: four complementary committor projections",
    fontsize=16,
)
TAI_Tools.save_fig_pdf_and_png(
    fig_summary,
    "HG_mechanistic_four_projection_summary",
    output_path=OUTPUT_DIR,
)
plt.show()


In [ ]:
fe_contour_levels = [2, 5, 10, 15, 20, 30, 40,43]
v_min_max_fe      = (0, 50)          # colour scale for ΔF
smooth_sigma_contour = 1
smooth_sigma_image = 0.0
ncols, nrows = 2, 2
fig, axarr = plt.subplots(
    nrows, ncols, figsize=(ncols * 6, nrows * 5),
    # gridspec_kw={"wspace": 0.4, "hspace": 0.5},
)
axarr = axarr.flatten()
fontsize_labels = 15
fontsize_cbar = 15
descriptors_show = [0,4]

n_bins_2d = (int(N_BINS_DIM[descriptors_show[0]]), int(N_BINS_DIM[descriptors_show[1]]))

# Use actual data range for extent so empty whitespace is avoided
actual_x_max = 1.1
actual_y_min = float(vis._desc_phys[:, descriptors_show[1]].min())
actual_y_max = float(vis._desc_phys[:, descriptors_show[1]].max())

# Update extent for this pair in PHYSICAL space (x = dim0, y = other_dim)
vis.dims_extent = [
    float(DESC_MIN[0]),         actual_x_max,
    actual_y_min,               actual_y_max,
]
vis.descriptor_dims = descriptors_show



ax_fe = axarr[0]

# Filled free-energy heatmap  (new method — uses vis._desc_phys, vis._w)
im_fe = vis.plot_rpe_free_energy(
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_fe,
    offset=True,
    v_min_max=v_min_max_fe,
    cmap="Blues",
    smooth_sigma=smooth_sigma_image
)

vis.plot_rpe_free_energy_contours(
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_fe,
    levels=fe_contour_levels,
    offset=True,
    colors="white", linewidths=0.8, alpha=0.7,
    clabel=True, clabel_fontsize=6,
    smooth_sigma=smooth_sigma_contour
)

ax_shot = axarr[1]
im_shot = vis.plot_shot_committor(
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_shot,
    v_min_max=v_min_max_q,
    cmap="Spectral",
    smooth_sigma=smooth_sigma_image
)

# Contour overlay: data-based shot committor (dashed grey)
vis.plot_shot_committor_contours(
    descriptor_dims=    descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_shot,
    levels=contour_levels_q,
    colors="dimgray",
    linewidths=0.9,
    alpha=0.75,
    linestyles="dashed",
    smooth_sigma=smooth_sigma_contour,
    clabel=True,
)

ax_model = axarr[2]
im_model = vis.plot_q_model_projection(
    model,
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_model,
    v_min_max=v_min_max_proj_q,
    cmap="Spectral",
    logit=True,
    smooth_sigma=smooth_sigma_image
)
vis.plot_q_model_projection_contours(
    model,
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_model,
    levels=contour_levels_q,
    logit=True,
    colors="black",
    linewidths=0.7,
    alpha=0.6,
    linestyles="solid",
    clabel=True,
    clabel_fontsize=6,
    smooth_sigma=smooth_sigma_contour
)


ax_slice = axarr[3]

im_slice = vis.plot_q_slice_heatmap(
    model,
    descriptor_dims=descriptors_show,
    n_bins_2d=n_bins_2d,
    ax=ax_slice,
    v_min_max=v_min_max_proj_q,
    cmap="Spectral",
    logit=True,
)
vis.plot_q_contours(
    model,
    ax=ax_slice,
    descriptor_dims=descriptors_show,
    n_bins_2d=200,
    levels=contour_levels_q,
    colors="black",
    linewidths=0.8,
    alpha=0.6,
    smooth_sigma=smooth_sigma_contour
)
# vlines span full y-axis regardless of physical y-range (xaxis transform)

# ---- colorbar labels for each panel ----
cbar_labels = [

    r"$\Delta F(\xi_1,\xi_2)$ [$k_{\mathrm{B}}T$]",
    r"$q_{\mathrm{shot}}(\xi_1,\xi_2)=\ln(n_B/n_A)$",
    r"$\langle q(x|\theta)\rangle_{\xi_1,\xi_2}$",
    r"$\langle q(x|\theta)\rangle_{\xi_1,\xi_2}$",

    # r"$\Delta F = -\ln(\rho_{\rm RPE})$ [$k_B T$]",
    # r"$\ln (n_B/n_A)$",
    # r"$q(x|\theta)$",
    # r"$q_{\rm model}$",
]

cbar_labels = [
    r"$\Delta F(\xi_i,\xi_j)$ [$k_{\mathrm{B}}T$]",
    r"$\bar{q}_{\rm shot}(\xi_i,\xi_j)=\ln(n_B/n_A)$",
    r"$\bar{q}_{\theta}(\xi_i,\xi_j)$",
    r"$q_{slice,\theta}(\xi_i,\xi_j)$",
]

titles = [
    "RPE free energy",
    "Shot committor",
    "Projected model committor",
    "Model committor slice",
]
titles = [""]*len(titles) 

ims = [im_fe, im_shot, im_model, im_slice]
axes = [ax_fe, ax_shot, ax_model, ax_slice]

for i, ax in enumerate(axes):

    ax.vlines(
        [STATE_A, STATE_B], 0, 1,
        transform=ax.get_xaxis_transform(),
        linewidths=1.5,
        colors="black",
        linestyles="--",
        zorder=10,
    )

    ax.set_xlim(float(DESC_MIN[descriptors_show[0]]), actual_x_max)
    ax.set_ylim(actual_y_min, actual_y_max)

    ax.set_xlabel(
        f"{DESCRIPTOR_LABELS[descriptors_show[0]]} "
        f"{DESCRIPTOR_UNITS[descriptors_show[0]]}",
        fontsize=fontsize_labels,
    )

    ax.set_ylabel(
        f"{DESCRIPTOR_LABELS[descriptors_show[1]]} "
        f"{DESCRIPTOR_UNITS[descriptors_show[1]]}",
        fontsize=fontsize_labels,
    )

    ax.set_title(titles[i], fontsize=12)

    cbar = fig.colorbar(
        ims[i],
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label(cbar_labels[i], fontsize=fontsize_cbar)
    cbar.ax.tick_params(labelsize=9)

from matplotlib.transforms import ScaledTranslation

labels = ["A", "B", "C","D"]

offset_x = -15  # pixels
offset_y = 5   # pixels

for i, ax in enumerate(axarr):

    trans = ax.transAxes + ScaledTranslation(
        offset_x / fig.dpi,
        offset_y / fig.dpi,
        fig.dpi_scale_trans
    )

    ax.text(
        0, 1,
        labels[i],
        transform=trans,
        fontsize=20,
        fontweight="bold",
        va="bottom",
        ha="right",
    )



fig.tight_layout()

TAI_Tools.save_fig_pdf_and_png(
    fig,
    "HG_projections_and_slices",
    output_path=OUTPUT_DIR,
)

plt.show()
fig.tight_layout()
TAI_Tools.save_fig_pdf_and_png(fig, "HG_projections_and_slices", output_path=OUTPUT_DIR)
plt.show()